# 06. Occupancy Grid Mapping

Occupancy grid는 지도를 작은 cell로 나누고 각 cell이 점유되었을 확률을 추정한다.

$$p(m_i\mid z_{1:t},x_{1:t})$$

계산 안정성을 위해 log-odds를 쓴다.

$$l_i = \log\frac{p(m_i)}{1-p(m_i)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Inverse Sensor Model로 log-odds 업데이트

레이저 ray가 지나간 cell은 free, endpoint 근처는 occupied로 업데이트한다.

In [ ]:
np.random.seed(10)
size=80
true_map=np.zeros((size,size),dtype=bool)
true_map[15:65,15]=True; true_map[15:65,65]=True; true_map[15,15:66]=True; true_map[65,15:66]=True
true_map[35:45,35:50]=True
logodds=np.zeros((size,size))
lo_occ=np.log(0.75/0.25); lo_free=np.log(0.35/0.65)
poses=np.array([[25,25,0.0],[35,25,0.2],[45,28,0.4],[55,35,1.0],[55,50,2.0],[40,55,3.0]])
angles=np.deg2rad(np.linspace(-80,80,33)); zmax=45

def cast(pose,ang):
    th=pose[2]+ang
    cells=[]
    for r in np.linspace(0,zmax,180):
        x=int(round(pose[0]+r*np.cos(th))); y=int(round(pose[1]+r*np.sin(th)))
        if x<0 or x>=size or y<0 or y>=size:
            return cells, None
        cells.append((y,x))
        if true_map[y,x]:
            return cells[:-1], (y,x)
    return cells, None

for pose in poses:
    for a in angles:
        free, occ = cast(pose,a)
        for y,x in free[::2]:
            logodds[y,x] += lo_free
        if occ is not None:
            y,x=occ; logodds[y,x] += lo_occ
logodds=np.clip(logodds,-5,5)
prob=1/(1+np.exp(-logodds))

fig, axes=plt.subplots(1,3,figsize=(14,5))
axes[0].imshow(true_map,cmap='gray_r',origin='lower'); axes[0].set_title('true map')
axes[1].imshow(prob,cmap='gray_r',origin='lower',vmin=0,vmax=1); axes[1].set_title('estimated occupancy probability')
im=axes[2].imshow(logodds,cmap='coolwarm',origin='lower',vmin=-5,vmax=5); axes[2].set_title('log-odds map')
for ax in axes:
    ax.scatter(poses[:,0],poses[:,1],s=30,color='#E85D24')
    ax.set_xticks([]); ax.set_yticks([])
plt.colorbar(im,ax=axes[2],fraction=0.046)
plt.tight_layout(); plt.savefig('assets/06_occupancy_grid.png',dpi=150,bbox_inches='tight'); plt.show()
print('occupied cells estimated:', int((prob>0.65).sum()))
print('free cells estimated:', int((prob<0.35).sum()))

## 2. 확률과 log-odds 변환

log-odds는 반복 업데이트에서 곱셈을 덧셈으로 바꿔준다.

In [ ]:
p=np.linspace(0.01,0.99,300)
l=np.log(p/(1-p))
fig, ax=plt.subplots(figsize=(7,4))
ax.plot(p,l,color='#534AB7',lw=2.5)
ax.axhline(0,color='gray',ls='--'); ax.axvline(0.5,color='gray',ls='--')
ax.set_xlabel('p(occupied)'); ax.set_ylabel('log odds')
ax.grid(alpha=0.25); ax.set_title('probability <-> log odds')
plt.savefig('assets/06_log_odds_curve.png',dpi=150,bbox_inches='tight'); plt.show()

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Occupancy grid | cell별 점유 확률 지도 | Ch.9 Occupancy Grid Mapping |
| Inverse sensor model | 관측에서 cell 점유 업데이트 | mapping 핵심 |
| Log-odds | 반복 업데이트 안정화 | 실전 구현 표준 |